In [1]:
import polars as pl
#import numpy as np
#import xgboost as xgb
#from sklearn.metrics import mean_absolute_error, root_mean_squared_error

#import geopandas as gpd
#import folium
#import matplotlib.colors as mcolors
#import matplotlib.pyplot as plt
#from branca.colormap import linear

INPUT_PATH  = "stgcn_dataset/node_features_X.parquet"
INPUT_PATH2 = "stgcn_dataset/node_features_X_with_airport.parquet"
OUTPUT_PATH = "stgcn_dataset/node_features_X_assignment3.parquet"

In [2]:
# Used to index a row
ID_COLS = [
    "time_bin",
    "LocationID",
    "node_index"
]

# Used as a standalone feature (NO ROLLING)
STATIC_COLS = [
    "zone_area_sqkm",
    "dist_to_center_km",
    "rolling_tip_pct",
    "rolling_avg_fare",
    "rolling_peak_ratio",
    "is_holiday",
    "hour",
    "weekday",
    "day_of_month",
    "month",
    "day_of_year",
    "year",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "month_sin",
    "month_cos",
    "doy_sin",
    "doy_cos",
    "is_airport_zone",
]

# Need lag for [0, 1, 2, 3, 23, 167]
BASE_COLS = [
    "demand",
    "revenue_total",
    "revenue_fare",
    "revenue_tip",
]

# Need lag for [-1, 0, 1, 2, 3, 23, 167]
FUTURE_INCL_COLS = [
    "temperature",
    "wind_speed",
    "precipitation",
    "ap_ewr_arrival_sched",
    "ap_ewr_departure_sched",
    "ap_ewr_total_sched",
    "ap_jfk_arrival_sched",
    "ap_jfk_departure_sched",
    "ap_jfk_total_sched",
    "ap_lga_arrival_sched",
    "ap_lga_departure_sched",
    "ap_lga_total_sched",
    "local_airport_arrival_sched",
    "local_airport_departure_sched",
    "local_airport_total_sched",
]

ALL_COLS = ID_COLS + STATIC_COLS + BASE_COLS + FUTURE_INCL_COLS

LAGS_BASE = [0, 1, 2, 3, 23, 167]
LAGS_FUTURE = [-1, 0, 1, 23, 167]

In [3]:
df = (
    pl.scan_parquet(INPUT_PATH2)
    .select(ALL_COLS)
    .filter(pl.col("year") >= 2023)
    .sort(["LocationID", "time_bin"])
)

lag_exprs_base = []
for col in BASE_COLS:
    for lag in LAGS_BASE:
        lag_exprs_base.append(
            pl.col(col).shift(lag).over("LocationID").alias(f"{col}_lag_{lag}")
        )

lag_exprs_weather = []
for col in FUTURE_INCL_COLS:
    for lag in LAGS_FUTURE:
        lag_exprs_weather.append(
            pl.col(col).shift(lag).over("LocationID").alias(f"{col}_lag_{lag}")
        )

target_exprs = [
    pl.col("demand").shift(-1).over("LocationID").alias("TARGET_demand"),
    pl.col("revenue_total").shift(-1).over("LocationID").alias("TARGET_revenue"),
]

df_final = df.with_columns(*lag_exprs_base)
df_final = df_final.with_columns(*lag_exprs_weather)
df_final = df_final.with_columns(*target_exprs)

df_final = df_final.with_columns(*[
    pl.when(pl.col(c).cast(pl.Float64).is_finite())
      .then(pl.col(c))
      .otherwise(None)
      .alias(c)
    for c in ["rolling_tip_pct", "rolling_avg_fare", "rolling_peak_ratio"]
])

df_final = df_final.drop_nulls()

df_final.sink_parquet(OUTPUT_PATH)
print(f"\nSaved XGBoost dataset to {OUTPUT_PATH}")


Saved XGBoost dataset to stgcn_dataset/node_features_X_assignment3.parquet


In [6]:
df_final = pl.read_parquet(OUTPUT_PATH)

In [7]:
df_final

time_bin,LocationID,node_index,zone_area_sqkm,dist_to_center_km,rolling_tip_pct,rolling_avg_fare,rolling_peak_ratio,is_holiday,hour,weekday,day_of_month,month,day_of_year,year,hour_sin,hour_cos,weekday_sin,weekday_cos,month_sin,month_cos,doy_sin,doy_cos,is_airport_zone,demand,revenue_total,revenue_fare,revenue_tip,temperature,wind_speed,precipitation,ap_ewr_arrival_sched,ap_ewr_departure_sched,ap_ewr_total_sched,ap_jfk_arrival_sched,ap_jfk_departure_sched,ap_jfk_total_sched,…,ap_jfk_total_sched_lag_-1,ap_jfk_total_sched_lag_0,ap_jfk_total_sched_lag_1,ap_jfk_total_sched_lag_23,ap_jfk_total_sched_lag_167,ap_lga_arrival_sched_lag_-1,ap_lga_arrival_sched_lag_0,ap_lga_arrival_sched_lag_1,ap_lga_arrival_sched_lag_23,ap_lga_arrival_sched_lag_167,ap_lga_departure_sched_lag_-1,ap_lga_departure_sched_lag_0,ap_lga_departure_sched_lag_1,ap_lga_departure_sched_lag_23,ap_lga_departure_sched_lag_167,ap_lga_total_sched_lag_-1,ap_lga_total_sched_lag_0,ap_lga_total_sched_lag_1,ap_lga_total_sched_lag_23,ap_lga_total_sched_lag_167,local_airport_arrival_sched_lag_-1,local_airport_arrival_sched_lag_0,local_airport_arrival_sched_lag_1,local_airport_arrival_sched_lag_23,local_airport_arrival_sched_lag_167,local_airport_departure_sched_lag_-1,local_airport_departure_sched_lag_0,local_airport_departure_sched_lag_1,local_airport_departure_sched_lag_23,local_airport_departure_sched_lag_167,local_airport_total_sched_lag_-1,local_airport_total_sched_lag_0,local_airport_total_sched_lag_1,local_airport_total_sched_lag_23,local_airport_total_sched_lag_167,TARGET_demand,TARGET_revenue
datetime[μs],i64,i64,f64,f64,f32,f64,f64,i32,i8,i8,i8,i8,i16,i32,f64,f64,f64,f64,f64,f64,f64,f64,i8,u32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f32
2023-01-07 23:00:00,1,0,7.343009,17.611394,0.1303,93.9036,0.649351,0,23,6,7,1,7,2023,-0.258819,0.965926,-0.781831,0.62349,0.5,0.866025,0.120208,0.992749,1,0,0.0,0.0,0.0,5.6,3.1,0.0,17.0,0.0,17.0,11.0,0.0,11.0,…,4.0,11.0,23.0,4.0,1.0,0.0,12.0,11.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,12.0,12.0,0.0,0.0,8.0,17.0,12.0,9.0,6.0,0.0,0.0,5.0,0.0,0.0,8.0,17.0,17.0,9.0,6.0,0,0.0
2023-01-08 00:00:00,1,0,7.343009,17.611394,0.13011,94.097386,0.641892,0,0,7,8,1,8,2023,0.0,1.0,-2.4493e-16,1.0,0.5,0.866025,0.137279,0.990532,1,0,0.0,0.0,0.0,5.0,4.1,0.0,8.0,0.0,8.0,4.0,0.0,4.0,…,0.0,4.0,11.0,0.0,2.0,0.0,0.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,0.0,0.0,0.0,8.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,17.0,0.0,0.0,0,0.0
2023-01-08 01:00:00,1,0,7.343009,17.611394,0.13011,94.097386,0.641892,0,1,7,8,1,8,2023,0.258819,0.965926,-2.4493e-16,1.0,0.5,0.866025,0.137279,0.990532,1,0,0.0,0.0,0.0,5.0,4.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,0.0,0.0,0,0.0
2023-01-08 02:00:00,1,0,7.343009,17.611394,0.13011,94.097386,0.641892,0,2,7,8,1,8,2023,0.5,0.866025,-2.4493e-16,1.0,0.5,0.866025,0.137279,0.990532,1,0,0.0,0.0,0.0,4.4,2.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0
2023-01-08 03:00:00,1,0,7.343009,17.611394,0.13011,94.097386,0.641892,0,3,7,8,1,8,2023,0.707107,0.707107,-2.4493e-16,1.0,0.5,0.866025,0.137279,0.990532,1,0,0.0,0.0,0.0,4.4,2.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-08-26 18:00:00,263,262,0.616541,3.626352,0.156474,14.779645,1.244457,0,18,2,26,8,238,2025,-1.0,-1.8370e-16,0.974928,-0.222521,-0.866025,-0.5,-0.816538,-0.577292,0,106,1635.98999,1346.880005,289.109985